# Lung CV Model — Tier 1 (multi-label texture classifier)

Fine-tunes the USCL ResNet-18 backbone (ultrasound-domain pretrained, already in hand) as a
**multi-label** classifier for the 4 texture-based pathologies visible in a single frame:
B-lines, consolidation (the sonographic sign of pneumonia), pleural effusion, pleural thickening.

Multi-label (independent sigmoid per class), not softmax, because these findings co-occur
clinically (e.g. consolidation + effusion together) — softmax would wrongly force them to
compete for probability mass.

Pneumothorax is deliberately **not** in this notebook — it's a motion pattern invisible in a
single frame, handled separately by a classical motion feature per the Models-Registry plan.

**Before running**: upload `Pulmonary/` (specifically `POCUS_extracted/` and `USCL_weights/`)
and `manifests/pulmonary_manifest.csv` to your Google Drive, preserving this relative structure,
e.g. under `MyDrive/POCUS-Project/`. Adjust `DRIVE_ROOT` in the config cell if your path differs.


## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q opencv-python-headless imageio scikit-learn down


In [2]:
import io
import json
import zipfile
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import imageio.v2 as imageio
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as T
from PIL import Image
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader

DRIVE_ROOT = Path('/content/drive/MyDrive/POCUS-Project')
MANIFEST_PATH = DRIVE_ROOT / 'manifests' / 'pulmonary_manifest.csv'
DATA_ROOT = DRIVE_ROOT / 'Pulmonary' / 'POCUS_extracted'
USCL_CKPT = DRIVE_ROOT / 'Pulmonary' / 'USCL_weights' / 'checkpoint' / 'best_model.pth'

FINDING_COLS = ['finding_b_lines', 'finding_consolidation', 'finding_pleural_effusion', 'finding_pleural_thickening']
FRAMES_PER_CLIP = 8
IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 25
LR = 1e-3

# 3-candidate lung bake-off (see Models-Registry.md): section 9 loops over BACKBONES_TO_RUN =
# ['uscl', 'imagenet', 'usfm'] and trains+evaluates each in this one run. USFM's weights ARE
# verified downloadable (checked directly against the USFM GitHub repo's own config + loading
# code), unlike UltraSam for cardiac.

# USFM checkpoint: hosted on Google Drive by the paper's authors (CC-BY-NC 4.0 -- non-commercial,
# fine for this internship). Downloaded once via gdown, then cached in Drive so it isn't
# re-fetched every session.
USFM_GDRIVE_ID = '1KRwXZgYterH895Z8EpXpR1L1eSMMJo4q'
USFM_CKPT = DRIVE_ROOT / 'Pulmonary' / 'USFM_weights' / 'USFM_latest.pth'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)


def set_seed(seed=42):
    # Data augmentation (random crop/flip) and DataLoader shuffling aren't seeded by default,
    # so re-running the same cell gives different numbers each time -- this pins that down so
    # runs (and the 3-way comparison) are reproducible.
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


: 

In [3]:
!ls /content/drive/MyDrive/

: 

## 2. Load and filter the manifest

Drops rows already flagged by the manifest itself (`flag_do_not_use`, `flag_off_target_organ`),
and builds the multi-label target array from the 4 finding columns.


In [4]:
df = pd.read_csv(MANIFEST_PATH)
df = df[~df['flag_do_not_use'] & ~df['flag_off_target_organ']].reset_index(drop=True)
df['filepath'] = df['filepath'].str.replace('\\', '/', regex=False)
df['labels'] = df[FINDING_COLS].astype(int).values.tolist()

print(f'{len(df)} clips/images after filtering')
print(df[FINDING_COLS].sum())


: 

In [5]:
label_counts = df[FINDING_COLS].sum()
n_no_finding = (df[FINDING_COLS].sum(axis=1) == 0).sum()

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar([c.replace("finding_", "") for c in FINDING_COLS], label_counts.values, color="steelblue")
ax.bar_label(bars)
ax.set_ylabel("Positive clips/images")
ax.set_title(f"Class balance ({len(df)} total clips, multi-label -- bars don't sum to {len(df)})")
plt.tight_layout()
plt.show()

print(f"Clips with no finding at all: {n_no_finding} ({n_no_finding/len(df):.0%})")
print("Note the effusion/thickening minority -- this is why pos_weight and threshold tuning show up later in training/eval.")


: 

## 3. Train/val split — K-fold, by clip, not by frame

Splitting after frame extraction would leak frames from the same clip into both train and val
(they're near-duplicates). Split the clips first, stratified on the original disease `class`
column as a reasonable proxy for balancing the finding labels too (true multi-label stratification
is overkill at this sample size).

Uses **`N_CV_SPLITS`-fold stratified K-fold** rather than repeated random splits: every clip is
held out exactly once across the `N_CV_SPLITS` folds -- none skipped, none double-counted. This
matters for section 10: pooling every fold's held-out predictions together gives one evaluation
that covers the **full 187 clips**, not just a ~56-clip slice of them the way a single split (or
even the mean of a few random repeated splits) would.

Each fold still holds out close to 30% of the clips (1/`N_CV_SPLITS` ~= 33% for 3 folds) -- same
reasoning as before for why that's safe here: the same ~131 training clips get analyzed from a
different angle each fold, so held-out size doesn't trade away training size the way it would
with a single fixed split.


In [6]:
from sklearn.model_selection import StratifiedKFold

N_CV_SPLITS = 3   # StratifiedKFold: every clip is held out exactly once across these folds, so
                   # pooling all out-of-fold predictions (section 10) covers the full 187 clips --
                   # a true partition, not a random repeated split. 3 folds keeps each fold's
                   # held-out share close to the 30% originally requested (1/3 ~= 33%).

# NOTE ON RUNTIME: section 9 trains len(BACKBONES_TO_RUN) x N_CV_SPLITS full models (25 epochs
# each). USFM (ViT-B) is already the slow candidate on a T4 -- raise N_CV_SPLITS for a tighter
# mean/std estimate if you have the time budget, or lower it (e.g. to 2, which still guarantees
# full coverage, just a coarser train/val ratio) if you don't.
skf = StratifiedKFold(n_splits=N_CV_SPLITS, shuffle=True, random_state=42)
cv_splits = list(skf.split(df, df['class']))
for i, (train_pos, val_pos) in enumerate(cv_splits):
    print(f'Fold {i+1}: {len(train_pos)} train clips, {len(val_pos)} val clips')


: 

## 4. Frame extraction

Videos are a mix of mp4/avi/mov/mpeg (OpenCV) and gif (OpenCV can't read these — falls back to
imageio). Images pass through as a single frame. Samples `FRAMES_PER_CLIP` evenly spaced frames
per clip — nearby frames in an ultrasound clip are near-identical for a texture-based task, so
more than ~8 buys little.


In [7]:
def extract_frames(filepath: Path, media_type: str, n_frames: int) -> list:
    if media_type == 'image':
        return [np.array(Image.open(filepath).convert('RGB'))]

    suffix = filepath.suffix.lower()
    if suffix == '.gif':
        frames = imageio.mimread(str(filepath))
        frames = [np.array(Image.fromarray(f).convert('RGB')) for f in frames]
    else:
        cap = cv2.VideoCapture(str(filepath))
        frames = []
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        cap.release()

    if not frames:
        return []
    idx = np.linspace(0, len(frames) - 1, min(n_frames, len(frames))).astype(int)
    return [frames[i] for i in idx]


: 

## 5. Dataset

`FrameCache` decodes every kept clip's frames **once**, over the full 187-clip manifest — frame
decoding is the slow part, and it doesn't need to be redone per split or per backbone, only which
clips are assigned to train vs val changes between them. Each frame carries a `clip_pos` back to
its row position in `df` so it can be reassigned to whichever fold needs it.

`ClipSubset` is a thin view over `FrameCache` restricted to one split's `train_pos`/`val_pos`
clip positions, with its own transform and clip ids renumbered to a contiguous `0..n-1` range (so
`aggregate_by_clip` — section 7 — works correctly per fold).


In [8]:
class FrameCache(Dataset):
    """Decodes every kept clip's frames once, over the full filtered manifest. Frames carry
    clip_pos = their row position in `df`, stable across every split -- so a fold only has to
    pick which positions go to train vs val, no re-decoding needed."""
    def __init__(self, manifest_df: pd.DataFrame, transform):
        self.transform = transform
        self.samples = []
        clip_pos = 0
        for _, row in manifest_df.iterrows():
            filepath = DATA_ROOT / row['filepath']
            if not filepath.exists():
                print('Missing file, skipping:', filepath)
                continue
            frames = extract_frames(filepath, row['media_type'], FRAMES_PER_CLIP)
            if not frames:
                continue
            label = np.array(row['labels'], dtype=np.float32)
            for frame in frames:
                self.samples.append((frame, label, clip_pos))
            clip_pos += 1
        self.num_clips = clip_pos

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        frame, label, clip_pos = self.samples[idx]
        image = self.transform(Image.fromarray(frame))
        return image, torch.from_numpy(label), clip_pos


class ClipSubset(Dataset):
    """View over FrameCache restricted to one split's clip positions. Renumbers clip ids to a
    contiguous 0..n-1 range (matching how many *clips*, not frames, are in this fold) so
    aggregate_by_clip works per fold. `global_positions[i]` maps a local clip id i back to its
    row position in `df`, so per-clip predictions can be pooled back into a full-dataset array
    (section 9/10) instead of only ever being seen inside their own fold. Every sample also
    carries a mask of all-ones -- this dataset's labels are always fully known; the mask only
    matters for LUS-BALD (below), which isn't."""
    def __init__(self, frame_cache: 'FrameCache', clip_positions, transform):
        self.transform = transform
        wanted = set(int(p) for p in clip_positions)
        self.global_positions = sorted(wanted)
        remap = {pos: i for i, pos in enumerate(self.global_positions)}
        self.samples = [(frame, label, remap[pos]) for frame, label, pos in frame_cache.samples if pos in wanted]
        self.num_clips = len(remap)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        frame, label, clip_idx = self.samples[idx]
        image = self.transform(Image.fromarray(frame))
        mask = torch.ones(len(FINDING_COLS), dtype=torch.float32)
        return image, torch.from_numpy(label), clip_idx, mask


# Normalization has to match what each backbone was actually pretrained with: USCL used
# mean/std=0.5/0.25 (not ImageNet stats), while a real torchvision ImageNet checkpoint expects
# ImageNet's own mean/std. USFM's exact stats aren't documented anywhere confirmable -- ImageNet
# stats used as a reasonable default, worth revisiting if USFM underperforms suspiciously.
NORM_STATS = {
    'uscl': ([0.5, 0.5, 0.5], [0.25, 0.25, 0.25]),
    'imagenet': ([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    'usfm': ([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    'efficientnet_b0': ([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
}


def make_transforms(norm_mean, norm_std):
    train_transform = T.Compose([
        T.Resize((IMG_SIZE, IMG_SIZE)),
        T.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0), ratio=(0.8, 1.25)),
        T.RandomHorizontalFlip(),
        # Mild rotation (probe angle varies a bit in practice) and brightness/contrast jitter
        # (different machines use different gain/TGC settings). More synthetic variety per real
        # clip is one of the cheaper levers against overfitting on ~131 training clips per split.
        T.RandomRotation(15),
        T.ColorJitter(brightness=0.2, contrast=0.2),
        T.ToTensor(),
        T.Normalize(mean=norm_mean, std=norm_std),
    ])
    val_transform = T.Compose([
        T.Resize((IMG_SIZE, IMG_SIZE)),
        T.ToTensor(),
        T.Normalize(mean=norm_mean, std=norm_std),
    ])
    return train_transform, val_transform


# Frame decoding (the slow part) happens once here, with a placeholder transform -- reused across
# every split and every backbone in section 9 via build_dataloaders() below.
_placeholder_t, _ = make_transforms(*NORM_STATS['imagenet'])
frame_cache = FrameCache(df, _placeholder_t)
print(f'Decoded {len(frame_cache)} frames from {frame_cache.num_clips} clips (decoded once, reused across every split and backbone)')


from torch.utils.data import ConcatDataset


def build_dataloaders(backbone_name, train_pos, val_pos):
    train_transform, val_transform = make_transforms(*NORM_STATS[backbone_name])
    train_dataset = ClipSubset(frame_cache, train_pos, train_transform)
    val_dataset = ClipSubset(frame_cache, val_pos, val_transform)
    # LUS-BALD is added to every fold's TRAIN loader only (see the markdown below this section) --
    # never held out, never validated on. train_dataset itself (returned separately) stays the
    # plain ClipSubset so pos_weight (section 7) can still read its .samples the same way as before.
    extra_train_dataset = ExtraLabelDataset(lus_bald_images, lus_bald_labels, lus_bald_masks, train_transform)
    combined_train_dataset = ConcatDataset([train_dataset, extra_train_dataset])
    train_loader = DataLoader(combined_train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
    return train_dataset, train_loader, val_dataset, val_loader


: 

### Supplementary training-only data — LUS-BALD (B-lines only, no negatives)

[Orvile's Kaggle "Lung Ultrasound Imaging Dataset for Accurate Detection and Localization of
B-line Artifacts"](https://www.kaggle.com/datasets/orvile/lung-ultrasound-imaging-dataset)
(CC BY 4.0), 401 images from 255 patients in Uganda -- a genuinely different population from the
POCUS Atlas/Butterfly sources `df` is built from. Inspecting the actual zip (not just the Kaggle
description) turned up two things that shape how this can be used:

- Its `test` split (70 images) ships with **no label files at all** -- unusable here.
- Every one of the 331 labeled images (`train` + `val`) is **B-lines positive** -- there isn't a
  single negative example in this dataset, and nothing at all is known about the other 3 findings
  for these images.

That means it can't function as its own independent train/val/test set (no negatives to learn or
validate against). Instead: **all 331 images are added to every fold's training set only** (never
held out, never validated on -- validation/test stay 100% `df`-sourced, exactly as before), and
each carries a **mask** alongside its label so the training loss only scores the column actually
known for that image (b_lines) and skips the other 3, rather than guessing them as negative.
`pos_weight` (section 7) is recomputed to include LUS-BALD's contribution to the b_lines column
only, since the other 3 columns' true counts are unaffected by images we have no information on.

**Before running**: upload `Pulmonary/LUS-BALD/LUS-BALD.zip` to Drive, same relative path as
everything else.


In [9]:
LUS_BALD_ZIP = DRIVE_ROOT / 'Pulmonary' / 'LUS-BALD' / 'LUS-BALD.zip'
LUS_BALD_ROOT = 'Lung Ultrasound Imaging Dataset for Accurate Detection and Localization of B-line Artifacts/LUS-BALD/LUS-BALD'


class ExtraLabelDataset(Dataset):
    """Wraps an external image source with ground truth for only a SUBSET of FINDING_COLS (here:
    only b_lines -- see the markdown above). Each sample carries a mask alongside its label so
    masked_bce_loss (section 7) can skip whichever columns aren't actually known for it, instead
    of guessing them as negative. Only ever added to a fold's TRAIN loader, never val."""
    def __init__(self, images, labels, masks, transform):
        self.images = images
        self.labels = labels
        self.masks = masks
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.transform(Image.fromarray(self.images[idx]))
        return image, torch.from_numpy(self.labels[idx]), -1, torch.from_numpy(self.masks[idx])


def load_lus_bald(zip_path, root_prefix):
    """Loads every image with a non-empty label file from LUS-BALD's train+val splits (its own
    test split ships with no labels at all, so it's unusable here). Every loaded image is B-lines
    positive by construction -- no negatives exist anywhere in this dataset."""
    images = []
    with zipfile.ZipFile(zip_path) as zf:
        names = set(zf.namelist())
        label_names = sorted(n for n in names if '/labels/' in n and n.endswith('.txt')
                              and (f'{root_prefix}/train/' in n or f'{root_prefix}/val/' in n))
        for label_name in label_names:
            stem = label_name.rsplit('/', 1)[-1].rsplit('.', 1)[0]
            split_dir = label_name.rsplit('/labels/', 1)[0]
            image_name = next((f'{split_dir}/images/{stem}{ext}' for ext in ('.png', '.jpg', '.jpeg')
                                if f'{split_dir}/images/{stem}{ext}' in names), None)
            if image_name is None:
                print('No matching image for label, skipping:', label_name)
                continue
            img = np.array(Image.open(io.BytesIO(zf.read(image_name))).convert('RGB'))
            images.append(img)

    n = len(images)
    b_lines_idx = FINDING_COLS.index('finding_b_lines')
    labels = np.zeros((n, len(FINDING_COLS)), dtype=np.float32)
    labels[:, b_lines_idx] = 1.0
    masks = np.zeros((n, len(FINDING_COLS)), dtype=np.float32)
    masks[:, b_lines_idx] = 1.0
    return images, labels, masks


lus_bald_images, lus_bald_labels, lus_bald_masks = load_lus_bald(LUS_BALD_ZIP, LUS_BALD_ROOT)
print(f'Loaded {len(lus_bald_images)} LUS-BALD images (train+val splits only -- test split has no labels)')


: 

## 6. Backbone — USCL vs USFM ViT-B

`ResNetUSCL` is copied from the USCL repo's own `resnet_uscl.py`. Per the checkpoint's own eval
recipe: discard the `linear`/`fc` keys when loading (trained for USCL's contrastive pretext task,
not our 4-label problem) — keep only the `features.*` backbone weights.

**USFM** is a real ViT-B/16 pretrained on 2M+ multi-organ ultrasound images (openmedlab/USFM,
CC-BY-NC 4.0). Its full repo depends on mmengine/mmseg/Hydra for the *segmentation* path, but the
*classification* path (`build_vit`) doesn't need any of that — the `VisionTransformer` class below
is copied directly from `usdsgen/modules/backbone/vision_transformer.py`, and the exact
architecture hyperparameters (patch_size=16, embed_dim=768, depth=12, num_heads=12, relative
position bias, no absolute position embedding) come straight from the repo's own
`configs/model/Cls/vit.yaml` — not guessed. The `remap_pretrained_keys_vit` function (also copied,
from `usdsgen/utils/modelutils.py`) handles expanding the checkpoint's shared relative-position-bias
into per-layer ones and interpolating it if our input resolution differs from pretraining.

Caveat flagged honestly: the exact input normalization stats USFM was pretrained with aren't
explicitly documented anywhere I could confirm — using ImageNet stats as a reasonable default,
noted in the code as an assumption worth revisiting if USFM underperforms suspiciously.

**ImageNet dropped from the active comparison**: it was tested earlier and actually tied/beat USCL
on most findings (a real, useful result to note when presenting this work), but the ultrasound-vs-
generic-photo-pretraining question has been answered for now. `build_model('imagenet')` still works
below if you want to bring it back — it's just not in `BACKBONES_TO_RUN` in section 9 anymore.

**Fine-tuning recipe — linear probing, not partial unfreezing**: `build_model` freezes the entire
pretrained backbone for every candidate and trains only the new head. An earlier version also
unfroze the last block/stage (~4.7M params for USCL/ImageNet's last ResNet block, ~7M+ for USFM's
last ViT block), which is a severe capacity mismatch against ~131 training clips/split (~1000
frames) — millions of trainable parameters chasing that little data overfits regardless of
dropout/weight_decay/augmentation layered on top. Freezing everything but the head cuts trainable
parameters to a few thousand, matching the model's adaptable capacity to the amount of data
actually available. If this underperforms (underfitting instead of overfitting), the next lever is
discriminative learning rates on a partially-unfrozen block, not going back to a fully-unfrozen one.

Below, `build_model(backbone_name)` builds whichever one you ask for. Section 9 calls it once per
candidate in a loop, so this whole notebook trains and compares both in a single run.


In [10]:
import math
from functools import partial

import numpy as np
from scipy.interpolate import RegularGridInterpolator as RGI
from timm.models.layers import DropPath, to_2tuple, trunc_normal_


# --- Everything below is copied from openmedlab/USFM's usdsgen/modules/backbone/vision_transformer.py
# and usdsgen/utils/modelutils.py (classification path only -- no mmseg/Hydra dependency needed) ---

class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None, act_layer=nn.GELU, drop=0.0):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = act_layer()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x


class Attention(nn.Module):
    def __init__(self, dim, num_heads=8, qkv_bias=False, qk_scale=None, attn_drop=0.0, proj_drop=0.0,
                 window_size=None, attn_head_dim=None):
        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // num_heads
        if attn_head_dim is not None:
            head_dim = attn_head_dim
        all_head_dim = head_dim * self.num_heads
        self.scale = qk_scale or head_dim ** -0.5

        self.qkv = nn.Linear(dim, all_head_dim * 3, bias=False)
        if qkv_bias:
            self.q_bias = nn.Parameter(torch.zeros(all_head_dim))
            self.v_bias = nn.Parameter(torch.zeros(all_head_dim))
        else:
            self.q_bias = None
            self.v_bias = None

        if window_size:
            self.window_size = window_size
            self.num_relative_distance = (2 * window_size[0] - 1) * (2 * window_size[1] - 1) + 3
            self.relative_position_bias_table = nn.Parameter(torch.zeros(self.num_relative_distance, num_heads))
            coords_h = torch.arange(window_size[0])
            coords_w = torch.arange(window_size[1])
            coords = torch.stack(torch.meshgrid([coords_h, coords_w]))
            coords_flatten = torch.flatten(coords, 1)
            relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]
            relative_coords = relative_coords.permute(1, 2, 0).contiguous()
            relative_coords[:, :, 0] += window_size[0] - 1
            relative_coords[:, :, 1] += window_size[1] - 1
            relative_coords[:, :, 0] *= 2 * window_size[1] - 1
            relative_position_index = torch.zeros(size=(window_size[0] * window_size[1] + 1,) * 2, dtype=relative_coords.dtype)
            relative_position_index[1:, 1:] = relative_coords.sum(-1)
            relative_position_index[0, 0:] = self.num_relative_distance - 3
            relative_position_index[0:, 0] = self.num_relative_distance - 2
            relative_position_index[0, 0] = self.num_relative_distance - 1
            self.register_buffer("relative_position_index", relative_position_index)
        else:
            self.window_size = None
            self.relative_position_bias_table = None
            self.relative_position_index = None

        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(all_head_dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x, rel_pos_bias=None):
        B, N, C = x.shape
        qkv_bias = None
        if self.q_bias is not None:
            qkv_bias = torch.cat((self.q_bias, torch.zeros_like(self.v_bias, requires_grad=False), self.v_bias))
        qkv = F.linear(input=x, weight=self.qkv.weight, bias=qkv_bias)
        qkv = qkv.reshape(B, N, 3, self.num_heads, -1).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        q = q * self.scale
        attn = q @ k.transpose(-2, -1)

        if self.relative_position_bias_table is not None:
            relative_position_bias = self.relative_position_bias_table[self.relative_position_index.view(-1)].view(
                self.window_size[0] * self.window_size[1] + 1, self.window_size[0] * self.window_size[1] + 1, -1)
            relative_position_bias = relative_position_bias.permute(2, 0, 1).contiguous()
            attn = attn + relative_position_bias.unsqueeze(0)
        if rel_pos_bias is not None:
            attn = attn + rel_pos_bias

        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        x = (attn @ v).transpose(1, 2).reshape(B, N, -1)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x


class Block(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4.0, qkv_bias=False, qk_scale=None, drop=0.0, attn_drop=0.0,
                 drop_path=0.0, init_values=None, act_layer=nn.GELU, norm_layer=nn.LayerNorm,
                 window_size=None, attn_head_dim=None):
        super().__init__()
        self.norm1 = norm_layer(dim)
        self.attn = Attention(dim, num_heads=num_heads, qkv_bias=qkv_bias, qk_scale=qk_scale,
                               attn_drop=attn_drop, proj_drop=drop, window_size=window_size, attn_head_dim=attn_head_dim)
        self.drop_path = DropPath(drop_path) if drop_path > 0.0 else nn.Identity()
        self.norm2 = norm_layer(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = Mlp(in_features=dim, hidden_features=mlp_hidden_dim, act_layer=act_layer, drop=drop)

        if init_values is not None:
            self.gamma_1 = nn.Parameter(init_values * torch.ones(dim), requires_grad=True)
            self.gamma_2 = nn.Parameter(init_values * torch.ones(dim), requires_grad=True)
        else:
            self.gamma_1, self.gamma_2 = None, None

    def forward(self, x, rel_pos_bias=None):
        if self.gamma_1 is None:
            x = x + self.drop_path(self.attn(self.norm1(x), rel_pos_bias=rel_pos_bias))
            x = x + self.drop_path(self.mlp(self.norm2(x)))
        else:
            x = x + self.drop_path(self.gamma_1 * self.attn(self.norm1(x), rel_pos_bias=rel_pos_bias))
            x = x + self.drop_path(self.gamma_2 * self.mlp(self.norm2(x)))
        return x


class PatchEmbed(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=768):
        super().__init__()
        img_size = to_2tuple(img_size)
        patch_size = to_2tuple(patch_size)
        num_patches = (img_size[1] // patch_size[1]) * (img_size[0] // patch_size[0])
        self.patch_shape = (img_size[0] // patch_size[0], img_size[1] // patch_size[1])
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = num_patches
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x, **kwargs):
        B, C, H, W = x.shape
        assert H == self.img_size[0] and W == self.img_size[1], \
            f"Input image size ({H}*{W}) doesn't match model ({self.img_size[0]}*{self.img_size[1]})."
        x = self.proj(x).flatten(2).transpose(1, 2)
        return x


class VisionTransformer(nn.Module):
    """BEiT-style ViT, copied from USFM's vision_transformer.py (VisionTransformer with support for
    relative position bias, no absolute position embedding -- matches configs/model/Cls/vit.yaml)."""

    def __init__(self, img_size=224, patch_size=16, in_chans=3, num_classes=1000, embed_dim=768, depth=12,
                 num_heads=12, mlp_ratio=4.0, qkv_bias=False, qk_scale=None, drop_rate=0.0, attn_drop_rate=0.0,
                 drop_path_rate=0.0, norm_layer=nn.LayerNorm, init_values=None, use_abs_pos_emb=True,
                 use_rel_pos_bias=False, use_shared_rel_pos_bias=False, use_mean_pooling=True,
                 init_scale=0.001, **kwargs):
        super().__init__()
        self.num_classes = num_classes
        self.num_features = self.embed_dim = embed_dim
        self.patch_size = patch_size
        self.in_chans = in_chans
        self.patch_embed = PatchEmbed(img_size=img_size, patch_size=patch_size, in_chans=in_chans, embed_dim=embed_dim)
        num_patches = self.patch_embed.num_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim)) if use_abs_pos_emb else None
        self.pos_drop = nn.Dropout(p=drop_rate)

        self.rel_pos_bias = None  # use_shared_rel_pos_bias=False in USFM's Cls config -- per-layer bias instead
        self.use_rel_pos_bias = use_rel_pos_bias

        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, depth)]
        self.blocks = nn.ModuleList([
            Block(dim=embed_dim, num_heads=num_heads, mlp_ratio=mlp_ratio, qkv_bias=qkv_bias, qk_scale=qk_scale,
                  drop=drop_rate, attn_drop=attn_drop_rate, drop_path=dpr[i], norm_layer=norm_layer,
                  init_values=init_values, window_size=self.patch_embed.patch_shape if use_rel_pos_bias else None)
            for i in range(depth)
        ])
        self.norm = nn.Identity() if use_mean_pooling else norm_layer(embed_dim)
        self.fc_norm = norm_layer(embed_dim) if use_mean_pooling else None
        self.head = nn.Linear(embed_dim, num_classes) if num_classes > 0 else nn.Identity()

        if self.pos_embed is not None:
            trunc_normal_(self.pos_embed, std=0.02)
        trunc_normal_(self.cls_token, std=0.02)
        if num_classes > 0:
            trunc_normal_(self.head.weight, std=0.02)
        self.apply(self._init_weights)
        self._fix_init_weight()
        if num_classes > 0:
            self.head.weight.data.mul_(init_scale)
            self.head.bias.data.mul_(init_scale)

    def _fix_init_weight(self):
        for layer_id, layer in enumerate(self.blocks):
            layer.attn.proj.weight.data.div_(math.sqrt(2.0 * (layer_id + 1)))
            layer.mlp.fc2.weight.data.div_(math.sqrt(2.0 * (layer_id + 1)))

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)

    def get_num_layers(self):
        return len(self.blocks)

    def forward_features(self, x):
        x = self.patch_embed(x)
        batch_size, seq_len, _ = x.size()
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        if self.pos_embed is not None:
            x = x + self.pos_embed
        x = self.pos_drop(x)
        for blk in self.blocks:
            x = blk(x, rel_pos_bias=None)
        x = self.norm(x)
        if self.fc_norm is not None:
            t = x[:, 1:, :]
            return self.fc_norm(t.mean(1))
        return x[:, 0]

    def forward(self, x):
        return self.head(self.forward_features(x))


def remap_pretrained_keys_vit(model, checkpoint_model):
    """Copied from USFM's modelutils.py -- expands the checkpoint's shared relative position bias
    into one per layer, and interpolates it if our input resolution differs from pretraining."""
    num_layers = model.get_num_layers()
    if "rel_pos_bias.relative_position_bias_table" in checkpoint_model:
        rel_pos_bias = checkpoint_model["rel_pos_bias.relative_position_bias_table"]
        for i in range(num_layers):
            checkpoint_model[f"blocks.{i}.attn.relative_position_bias_table"] = rel_pos_bias.clone()
        checkpoint_model.pop("rel_pos_bias.relative_position_bias_table")

    all_keys = list(checkpoint_model.keys())
    for key in all_keys:
        if "relative_position_index" in key:
            checkpoint_model.pop(key)
        if "relative_position_bias_table" in key and key in model.state_dict():
            rel_pos_bias = checkpoint_model[key]
            src_num_pos, num_attn_heads = rel_pos_bias.size()
            dst_num_pos, _ = model.state_dict()[key].size()
            if src_num_pos == dst_num_pos:
                continue
            dst_patch_shape = model.patch_embed.patch_shape
            num_extra_tokens = dst_num_pos - (dst_patch_shape[0] * 2 - 1) * (dst_patch_shape[1] * 2 - 1)
            src_size = int((src_num_pos - num_extra_tokens) ** 0.5)
            dst_size = int((dst_num_pos - num_extra_tokens) ** 0.5)
            print(f'Position interpolate for {key} from {src_size}x{src_size} to {dst_size}x{dst_size}')
            extra_tokens = rel_pos_bias[-num_extra_tokens:, :]
            rel_pos_bias = rel_pos_bias[:-num_extra_tokens, :]

            def geometric_progression(a, r, n):
                return a * (1.0 - r ** n) / (1.0 - r)

            left, right = 1.01, 1.5
            while right - left > 1e-6:
                q = (left + right) / 2.0
                gp = geometric_progression(1, q, src_size // 2)
                right, left = (q, left) if gp > dst_size // 2 else (right, q)
            dis = []
            cur = 1
            for i in range(src_size // 2):
                dis.append(cur)
                cur += q ** (i + 1)
            r_ids = [-_ for _ in reversed(dis)]
            x = r_ids + [0] + dis
            y = r_ids + [0] + dis
            t = dst_size // 2.0
            dx = np.arange(-t, t + 0.1, 1.0)
            dy = np.arange(-t, t + 0.1, 1.0)
            xi, yi = np.meshgrid(dx, dy, indexing="ij")
            points = np.array([xi.ravel(), yi.ravel()]).T
            all_rel_pos_bias = []
            for i in range(num_attn_heads):
                z = rel_pos_bias[:, i].view(src_size, src_size).float().numpy()
                f = RGI((x, y), z.T, method="cubic", bounds_error=False)
                all_rel_pos_bias.append(torch.Tensor(f(points).reshape(xi.shape)).contiguous().view(-1, 1))
            rel_pos_bias = torch.cat(all_rel_pos_bias, dim=-1)
            checkpoint_model[key] = torch.cat((rel_pos_bias, extra_tokens), dim=0)
    return checkpoint_model


class USFMFeatureExtractor(nn.Module):
    """Thin wrapper so USFM's ViT plugs into the same feature_extractor(x) -> (B, num_ftrs)
    interface as the ResNet backbones, matching what LungFindingClassifier expects."""
    def __init__(self, vit: VisionTransformer):
        super().__init__()
        self.vit = vit

    def forward(self, x):
        return self.vit.forward_features(x)


: 

In [11]:
class ResNetUSCL(nn.Module):
    def __init__(self, base_model='resnet18', out_dim=256):
        super().__init__()
        resnet = models.resnet18(weights=None)
        num_ftrs = resnet.fc.in_features
        self.features = nn.Sequential(*list(resnet.children())[:-1])
        self.linear = nn.Linear(num_ftrs, out_dim)


class LungFindingClassifier(nn.Module):
    def __init__(self, feature_extractor: nn.Module, num_ftrs: int, num_labels: int, dropout: float = 0.3):
        super().__init__()
        self.features = feature_extractor
        # Dropout on the pooled features -- the only regularization the new head had before,
        # which combined with ~131 training clips/split is exactly the overfitting the loss curves
        # show (train_loss -> ~0, val_loss climbing). Applied identically across all backbones.
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(num_ftrs, num_labels)

    def forward(self, x):
        h = self.features(x)
        h = h.flatten(1)
        h = self.dropout(h)
        return self.head(h)


def build_model(backbone_name):
    if backbone_name == 'uscl':
        backbone = ResNetUSCL(base_model='resnet18', out_dim=256)
        state_dict = torch.load(USCL_CKPT, map_location='cpu')
        backbone_dict = {k: v for k, v in state_dict.items() if not (k.startswith('l') or k.startswith('fc'))}
        missing, unexpected = backbone.load_state_dict(backbone_dict, strict=False)
        print('Missing keys:', missing)
        print('Unexpected keys:', unexpected)
        feature_extractor = backbone.features
        num_ftrs = backbone.linear.in_features

    elif backbone_name == 'imagenet':
        resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        num_ftrs = resnet.fc.in_features
        feature_extractor = nn.Sequential(*list(resnet.children())[:-1])

    elif backbone_name == 'efficientnet_b0':
        # Standard ImageNet-pretrained CNN, no domain-specific ultrasound backbone -- same pick
        # already made for the gallbladder task (see Models-Registry.md), reused here since USCL
        # and USFM (two genuinely different architecture families) converged to the same AUROC
        # range and the same weak spots, pointing at dataset size as the ceiling rather than
        # backbone choice. Kept in the bake-off structure anyway in case that read is wrong.
        effnet = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        num_ftrs = effnet.classifier[1].in_features  # 1280
        feature_extractor = nn.Sequential(effnet.features, effnet.avgpool)

    elif backbone_name == 'usfm':
        if not USFM_CKPT.exists():
            USFM_CKPT.parent.mkdir(parents=True, exist_ok=True)
            import gdown
            gdown.download(id=USFM_GDRIVE_ID, output=str(USFM_CKPT), quiet=False)
        # Exact hyperparameters from USFM's own configs/model/Cls/vit.yaml -- not guessed.
        vit = VisionTransformer(
            img_size=IMG_SIZE, patch_size=16, in_chans=3, num_classes=0,  # num_classes=0 -> just the pooled embedding, no head
            embed_dim=768, depth=12, num_heads=12, mlp_ratio=4.0, qkv_bias=True,
            attn_drop_rate=0.0, drop_path_rate=0.1, init_values=0.1,
            use_abs_pos_emb=False, use_rel_pos_bias=True, use_shared_rel_pos_bias=False, use_mean_pooling=True,
        )
        checkpoint = torch.load(USFM_CKPT, map_location='cpu')
        checkpoint = checkpoint.get('model', checkpoint)  # some checkpoints wrap the state_dict under a 'model' key
        checkpoint = remap_pretrained_keys_vit(vit, checkpoint)
        missing, unexpected = vit.load_state_dict(checkpoint, strict=False)
        print('Missing keys:', missing)
        print('Unexpected keys:', unexpected)
        feature_extractor = USFMFeatureExtractor(vit)
        num_ftrs = 768

    else:
        raise ValueError(f'Unknown backbone: {backbone_name}')

    model = LungFindingClassifier(feature_extractor, num_ftrs, num_labels=len(FINDING_COLS)).to(device)

    # Linear probing: freeze the ENTIRE pretrained backbone, train only the new head. Previously
    # the last block/stage was also unfrozen (~4.7M params for USCL/ImageNet's last ResNet block,
    # ~7M+ for USFM's last ViT block) -- against ~131 training clips (~1000 frames) per split, that
    # many trainable parameters chasing that little data is a capacity mismatch severe enough to
    # overfit regardless of dropout/weight_decay/augmentation on top. The whole point of a
    # pretrained backbone is that it already learned general visual structure from a huge amount of
    # data; recombining those features for 4 findings needs far fewer trainable parameters than
    # re-adapting a full residual block or transformer block does. Applied identically across every
    # backbone for a fair comparison.
    for name, param in model.named_parameters():
        param.requires_grad = name.startswith('head')

    trainable = [n for n, p in model.named_parameters() if p.requires_grad]
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Backbone: {backbone_name} | Trainable parameters: {trainable} ({n_trainable:,} total)')
    return model


: 

## 7. Train

`train_model(model, run_name, train_dataset, train_loader, val_loader)` runs the training loop
for one (backbone, split) run and plots its loss curve. Called once per backbone per CV split by
the nested loop in section 9.

Uses a cosine-decaying LR (flat LR for all 25 epochs otherwise keeps pushing the model to fit
training-set noise late in the run) and selects/reports the checkpoint by **clip-level** val
AUROC — frame-level predictions within a clip are averaged first (`aggregate_by_clip`) since
they aren't independent evidence.


In [12]:
def run_epoch(model, loader, criterion, optimizer, train: bool):
    model.train() if train else model.eval()
    total_loss = 0.0
    all_logits, all_labels, all_clip_ids = [], [], []
    with torch.set_grad_enabled(train):
        for images, labels, clip_ids, masks in loader:
            images, labels, masks = images.to(device), labels.to(device), masks.to(device)
            logits = model(images)
            loss = criterion(logits, labels, masks)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * images.size(0)
            all_logits.append(logits.detach().cpu())
            all_labels.append(labels.detach().cpu())
            all_clip_ids.append(clip_ids)
    avg_loss = total_loss / len(loader.dataset)
    logits = torch.cat(all_logits).numpy()
    labels = torch.cat(all_labels).numpy()
    clip_ids = torch.cat(all_clip_ids).numpy()
    return avg_loss, logits, labels, clip_ids


def predict(model, loader):
    model.eval()
    all_logits, all_labels, all_clip_ids = [], [], []
    with torch.no_grad():
        for images, labels, clip_ids, _ in loader:
            logits = model(images.to(device))
            all_logits.append(logits.cpu())
            all_labels.append(labels)
            all_clip_ids.append(clip_ids)
    return torch.cat(all_logits).numpy(), torch.cat(all_labels).numpy(), torch.cat(all_clip_ids).numpy()


def aggregate_by_clip(probs, labels, clip_ids):
    """Average frame-level probabilities within each clip before scoring. Frames from the same
    clip are near-duplicates, not independent evidence (same patient, same finding) -- scoring
    every frame separately both overstates the effective val set size and adds frame-to-frame
    noise to the metric. Every frame in a clip shares the same label, so any one of them
    represents the clip's ground truth."""
    n_clips = int(clip_ids.max()) + 1
    clip_probs = np.zeros((n_clips, probs.shape[1]))
    clip_labels = np.zeros((n_clips, labels.shape[1]))
    for c in range(n_clips):
        mask = clip_ids == c
        clip_probs[c] = probs[mask].mean(axis=0)
        clip_labels[c] = labels[mask][0]
    return clip_probs, clip_labels


def per_label_auroc(probs, labels):
    aucs = {}
    for i, name in enumerate(FINDING_COLS):
        if len(np.unique(labels[:, i])) < 2:
            continue  # AUROC undefined with only one class present in this split
        aucs[name] = roc_auc_score(labels[:, i], probs[:, i])
    return aucs


def train_model(model, run_name, train_dataset, train_loader, val_loader):
    # Class-weighted loss: effusion/thickening are rarer positives than b-lines/consolidation
    # in this dataset, so an unweighted loss underfits them. pos_weight tells the loss to
    # penalize a missed positive for a rare class more than for a common one. Computed from this
    # run's own train_dataset since which clips (and their finding balance) land in train changes
    # every split.
    train_labels = np.stack([label for _, label, _ in train_dataset.samples])
    pos_counts = train_labels.sum(axis=0)
    total_known = np.full(len(FINDING_COLS), len(train_labels), dtype=np.float32)

    # LUS-BALD supplements every fold's training set (added in build_dataloaders, not part of the
    # fold rotation) with extra B-lines-positive images -- see section 5's LUS-BALD markdown. Only
    # the b_lines column's counts include them: the other 3 columns' ground truth is genuinely
    # unknown for these images (masked_bce_loss below skips them, not guessed as negative), so
    # they don't belong in those columns' pos_weight either.
    b_lines_idx = FINDING_COLS.index('finding_b_lines')
    pos_counts[b_lines_idx] += len(lus_bald_images)
    total_known[b_lines_idx] += len(lus_bald_images)

    neg_counts = total_known - pos_counts
    pos_weight = torch.tensor(neg_counts / np.clip(pos_counts, 1, None), dtype=torch.float32).to(device)
    print('pos_weight per label (includes LUS-BALD\'s contribution to b_lines):', dict(zip(FINDING_COLS, pos_weight.cpu().numpy())))

    def masked_bce_loss(logits, labels, masks):
        # Per-element BCE, then average only over the entries whose mask says the label is
        # actually known -- LUS-BALD samples contribute to b_lines only, ClipSubset samples
        # (mask all-ones) behave exactly like the unmasked loss did before.
        loss_per_elem = F.binary_cross_entropy_with_logits(logits, labels, pos_weight=pos_weight, reduction='none')
        return (loss_per_elem * masks).sum() / masks.sum().clamp(min=1)

    criterion = masked_bce_loss
    # weight_decay: L2 regularization on top of dropout -- both target the same overfitting
    # problem (train_loss -> 0 while val_loss climbs), which is expected on ~131 clips/split.
    optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=1e-4)
    # Cosine LR decay: a flat LR for all 25 epochs keeps pushing the model to fit training-set
    # noise late in the run. Decaying it lets the model learn fast early and settle gently later
    # instead -- another lever against overfitting, alongside dropout/weight_decay.
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    ckpt_path = DRIVE_ROOT / 'Pulmonary' / f'lung_finding_classifier_{run_name}_best.pth'
    # Select the checkpoint by validation AUROC, not validation loss -- loss can keep improving
    # or worsening in ways that don't track the discriminative metrics we actually report.
    best_avg_auroc = -1.0
    train_loss_history, val_loss_history, val_auroc_history = [], [], []
    for epoch in range(EPOCHS):
        train_loss, _, _, _ = run_epoch(model, train_loader, criterion, optimizer, train=True)
        val_loss, val_logits, val_labels, val_clip_ids = run_epoch(model, val_loader, criterion, optimizer, train=False)
        scheduler.step()
        train_loss_history.append(train_loss)
        val_loss_history.append(val_loss)

        # Score at the clip level, not per-frame -- see aggregate_by_clip's docstring.
        val_probs = 1 / (1 + np.exp(-val_logits))
        clip_probs, clip_labels = aggregate_by_clip(val_probs, val_labels, val_clip_ids)
        aucs = per_label_auroc(clip_probs, clip_labels)
        avg_auroc = float(np.mean(list(aucs.values()))) if aucs else 0.0
        val_auroc_history.append(avg_auroc)
        auc_str = ', '.join(f'{k}={v:.3f}' for k, v in aucs.items())
        print(f'[{run_name}] Epoch {epoch+1:02d}/{EPOCHS} | train_loss={train_loss:.4f} val_loss={val_loss:.4f} lr={scheduler.get_last_lr()[0]:.2e} avg_auroc={avg_auroc:.3f} | {auc_str}')
        if avg_auroc > best_avg_auroc:
            best_avg_auroc = avg_auroc
            torch.save(model.state_dict(), ckpt_path)

    print(f'\n[{run_name}] Best avg val AUROC (clip-level):', best_avg_auroc)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(train_loss_history, label='train_loss')
    ax1.plot(val_loss_history, label='val_loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title(f'{run_name} — loss')
    ax1.legend()
    ax1.grid(alpha=0.3)
    # A widening gap between the two lines (val rising while train keeps falling) is overfitting --
    # expected risk on ~131 training clips per split, worth watching for when comparing backbones.

    ax2.plot(val_auroc_history, color='green')
    ax2.axhline(best_avg_auroc, color='gray', linestyle=':', linewidth=1)
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Val avg AUROC (clip-level)')
    ax2.set_title(f'{run_name} — val AUROC (checkpoint selection metric)')
    ax2.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    return ckpt_path


: 

## 8. Final evaluation

`evaluate_model(...)` computes per-label AUROC and F1 (at a 0.5 threshold, and at a per-class
tuned threshold) on one split's held-out val clips, prints them, and saves them to a small JSON
file (`results_{backbone}_split{i}.json`) so section 10 can average across every split.

All metrics are computed **per clip** (frame-level probabilities averaged within each clip first),
not per frame — a clip is the unit a clinician actually gets a prediction for, and scoring frames
independently would inflate the apparent val set size with near-duplicate evidence.

Per-split bar charts/confusion matrices were dropped here (with `N_CV_SPLITS` runs per backbone
that's a lot of near-duplicate figures) in favor of the single averaged view in section 10.


In [13]:
def evaluate_model(model, backbone_name, split_idx, ckpt_path, val_loader):
    run_name = f'{backbone_name}_split{split_idx}'
    model.load_state_dict(torch.load(ckpt_path))
    print("Run:", run_name)
    val_logits, val_labels, val_clip_ids = predict(model, val_loader)
    val_probs_frame = 1 / (1 + np.exp(-val_logits))
    # Score at the clip level, not per-frame -- see aggregate_by_clip's docstring in section 7.
    val_probs, val_labels = aggregate_by_clip(val_probs_frame, val_labels, val_clip_ids)
    val_preds_default = (val_probs > 0.5).astype(int)

    aucs = per_label_auroc(val_probs, val_labels)
    print("Per-label AUROC (clip-level):", aucs)
    print(chr(10) + "--- Fixed 0.5 threshold (what we reported before) ---")
    for i, name in enumerate(FINDING_COLS):
        f1 = f1_score(val_labels[:, i], val_preds_default[:, i], zero_division=0)
        acc = accuracy_score(val_labels[:, i], val_preds_default[:, i])
        print(f"{name}: F1={f1:.3f} Accuracy={acc:.3f}")

    # Per-class threshold tuning: pick the threshold that maximizes F1 on this val set instead of
    # assuming 0.5. Honest caveat: with only ~56 val clips per fold, tuning and evaluating on the
    # same fold is optimistic -- treat this as what is achievable with a better cutoff, not an
    # unbiased estimate. Section 10's pooled evaluation (full 187 clips) is the more trustworthy
    # number; finding more real data (ICLUS-DB/COVIDx-US/BEDLUS) is still the real fix for this gap.
    print(chr(10) + "--- Per-class tuned threshold ---")
    best_thresholds = {}
    tuned_f1 = {}
    tuned_acc = {}
    tuned_preds = np.zeros_like(val_preds_default)
    for i, name in enumerate(FINDING_COLS):
        best_f1, best_t = 0.0, 0.5
        for t in np.arange(0.05, 0.95, 0.05):
            preds_t = (val_probs[:, i] > t).astype(int)
            f1_t = f1_score(val_labels[:, i], preds_t, zero_division=0)
            if f1_t > best_f1:
                best_f1, best_t = f1_t, t
        best_thresholds[name] = best_t
        tuned_f1[name] = best_f1
        tuned_preds[:, i] = (val_probs[:, i] > best_t).astype(int)
        tuned_acc[name] = accuracy_score(val_labels[:, i], tuned_preds[:, i])
        print(f"{name}: best_threshold={best_t:.2f} F1={best_f1:.3f} Accuracy={tuned_acc[name]:.3f}")

    results = {
        'backbone': backbone_name,
        'split': split_idx,
        'auroc': aucs,
        'f1_tuned': tuned_f1,
        'accuracy_tuned': tuned_acc,
        'thresholds': best_thresholds,
    }
    results_path = DRIVE_ROOT / 'Pulmonary' / f'results_{run_name}.json'
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f'Saved results to {results_path}')
    # val_probs returned (not just saved to JSON) so section 9 can pool every fold's held-out
    # predictions into one full-dataset evaluation -- see ClipSubset.global_positions.
    return results, val_probs


: 

## 9. Run the full bake-off

Trains and evaluates both backbones (`uscl`, `usfm`) across every CV split, back-to-back in this
one run — no manual re-running needed. For each split, each backbone gets a fresh model trained
on that split's ~70% train clips and evaluated on its ~30% held-out clips; results are collected
into `all_results[backbone_name]` as a list (one dict per split) for section 10 to average.

This is `len(BACKBONES_TO_RUN) x N_CV_SPLITS` full training runs — with USFM (ViT-B) already
slower per epoch than USCL's ResNet-18 on a T4, expect this to take a while. Lower `N_CV_SPLITS`
in section 3 if this doesn't fit your session's time budget.


In [ ]:
BACKBONES_TO_RUN = ['efficientnet_b0']  # dropped uscl/usfm from the active run per the time-budget
# call -- both are still fully wired up in build_model above if you want to bring them back for
# a fuller comparison later.
all_results = {b: [] for b in BACKBONES_TO_RUN}
# Every clip's out-of-fold prediction, filled in as its fold runs -- by the end, every row is
# filled exactly once per backbone (StratifiedKFold guarantees full, non-overlapping coverage).
pooled_probs = {b: np.full((len(df), len(FINDING_COLS)), np.nan) for b in BACKBONES_TO_RUN}
pooled_labels = df[FINDING_COLS].to_numpy(dtype=int)

for split_idx, (train_pos, val_pos) in enumerate(cv_splits):
    print(f'\n{"#"*24} Fold {split_idx+1}/{N_CV_SPLITS} {"#"*24}')
    for backbone_name in BACKBONES_TO_RUN:
        run_name = f'{backbone_name}_split{split_idx}'
        print(f'\n{"="*20} {run_name} {"="*20}')
        # Seed varies per fold (so augmentation/init differ across folds) but stays fixed across
        # backbones within a fold, keeping the bake-off comparison fair.
        set_seed(42 + split_idx)
        train_dataset, train_loader, val_dataset, val_loader = build_dataloaders(backbone_name, train_pos, val_pos)
        model = build_model(backbone_name)
        ckpt_path = train_model(model, run_name, train_dataset, train_loader, val_loader)
        result, val_probs = evaluate_model(model, backbone_name, split_idx, ckpt_path, val_loader)
        all_results[backbone_name].append(result)
        pooled_probs[backbone_name][val_dataset.global_positions] = val_probs


: 

## 10. Compare candidates

Two complementary views:

1. **Mean ± std across folds** (from `all_results`) -- shows how much a single fold's number
   could vary depending on which clips it happened to hold out. If a backbone's list is short
   (e.g. you lowered `N_CV_SPLITS` or interrupted a run), it fills in whatever
   `results_{backbone}_split{i}.json` files it can find on Drive from earlier runs.
2. **Pooled out-of-fold evaluation** (from `pooled_probs`) -- every one of the 187 clips gets
   exactly one prediction, made by whichever fold's model didn't train on it, then all 187 are
   scored together as one evaluation set. This is the more statistically trustworthy number (full
   dataset, not a ~56-clip slice), but it only works for backbones actually trained *this session*
   -- raw per-clip probabilities aren't persisted to Drive the way the summarized metrics are, so
   a resumed/partial session can't reconstruct it the way view 1 can.


In [ ]:
def load_missing_splits(backbone_name, results_list):
    have = {r['split'] for r in results_list}
    for split_idx in range(N_CV_SPLITS):
        if split_idx in have:
            continue
        path = DRIVE_ROOT / 'Pulmonary' / f'results_{backbone_name}_split{split_idx}.json'
        if path.exists():
            results_list.append(json.load(open(path)))
    return results_list


def mean_std_per_label(results_list, metric):
    values = {name: [] for name in FINDING_COLS}
    for res in results_list:
        for name in FINDING_COLS:
            if name in res[metric]:
                values[name].append(res[metric][name])
    means = {name: (np.mean(v) if v else np.nan) for name, v in values.items()}
    stds = {name: (np.std(v) if v else np.nan) for name, v in values.items()}
    return means, stds


for b in BACKBONES_TO_RUN:
    all_results[b] = load_missing_splits(b, all_results.get(b, []))

usable = {b: res for b, res in all_results.items() if len(res) > 0}
if len(usable) < 1:
    print('
No backbone has any fold run yet -- go run section 9 first.')
else:
    # Works with any number of backbones (1 or more) -- earlier this only plotted with >=2, which
    # broke once BACKBONES_TO_RUN was cut down to a single candidate (efficientnet_b0).
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    x = np.arange(len(FINDING_COLS))
    width = 0.8 / len(usable)

    metrics = [('auroc', axes[0], 'AUROC'), ('f1_tuned', axes[1], 'F1 (tuned threshold)'),
               ('accuracy_tuned', axes[2], 'Accuracy (tuned threshold)')]
    summary = {}
    for metric, ax, title in metrics:
        for i, (b, res_list) in enumerate(usable.items()):
            means, stds = mean_std_per_label(res_list, metric)
            summary.setdefault(b, {})[metric] = means
            values = [means[n] for n in FINDING_COLS]
            errors = [stds[n] for n in FINDING_COLS]
            ax.bar(x + i * width, values, width, yerr=errors, capsize=3, label=f'{b} (n={len(res_list)})')
        ax.set_xticks(x + width * (len(usable) - 1) / 2)
        ax.set_xticklabels([n.replace('finding_', '') for n in FINDING_COLS], rotation=20)
        ax.set_ylim(0, 1)
        ax.axhline(0.9, color='red', linestyle='--', linewidth=1)
        ax.set_title(f'{title} (mean ± std across folds)')
        ax.legend()
    plt.tight_layout()
    plt.show()

    print('
Best per label (by mean F1 across folds):')
    for name in FINDING_COLS:
        best_b = max(summary, key=lambda b: summary[b]['f1_tuned'][name])
        print(f'  {name}: {best_b} (mean F1={summary[best_b]["f1_tuned"][name]:.3f})')

# --- Pooled out-of-fold evaluation: every clip scored exactly once, covering the full dataset ---
# Self-contained (doesn't reuse 'x'/'width' from the block above) since that block can be skipped
# (len(usable) < 1) independently of whether pooled coverage exists.
complete_pooled = {b: p for b, p in pooled_probs.items() if not np.isnan(p).any()}
if not complete_pooled:
    print('
No backbone has complete pooled coverage this session -- run all folds for at least one backbone to see this.')
else:
    print(f'
=== Pooled out-of-fold evaluation (all {len(df)} clips, each scored exactly once) ===')
    pooled_summary = {}
    for b, probs in complete_pooled.items():
        aucs_pooled = per_label_auroc(probs, pooled_labels)
        f1_pooled, acc_pooled, thresh_pooled = {}, {}, {}
        for i, name in enumerate(FINDING_COLS):
            best_f1, best_t = 0.0, 0.5
            for t in np.arange(0.05, 0.95, 0.05):
                f1_t = f1_score(pooled_labels[:, i], (probs[:, i] > t).astype(int), zero_division=0)
                if f1_t > best_f1:
                    best_f1, best_t = f1_t, t
            f1_pooled[name] = best_f1
            thresh_pooled[name] = best_t
            acc_pooled[name] = accuracy_score(pooled_labels[:, i], (probs[:, i] > best_t).astype(int))
        pooled_summary[b] = {'auroc': aucs_pooled, 'f1_tuned': f1_pooled, 'accuracy_tuned': acc_pooled}
        print(f'
[{b}]')
        for name in FINDING_COLS:
            print(f'  {name}: AUROC={aucs_pooled.get(name, float("nan")):.3f} '
                  f'F1={f1_pooled[name]:.3f} (threshold={thresh_pooled[name]:.2f}) Accuracy={acc_pooled[name]:.3f}')

    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    x = np.arange(len(FINDING_COLS))
    width = 0.8 / len(pooled_summary)
    metrics = [('auroc', axes[0], 'AUROC'), ('f1_tuned', axes[1], 'F1 (tuned threshold)'),
               ('accuracy_tuned', axes[2], 'Accuracy (tuned threshold)')]
    for metric, ax, title in metrics:
        for i, (b, res) in enumerate(pooled_summary.items()):
            values = [res[metric][n] for n in FINDING_COLS]
            ax.bar(x + i * width, values, width, label=b)
        ax.set_xticks(x + width * (len(pooled_summary) - 1) / 2)
        ax.set_xticklabels([n.replace('finding_', '') for n in FINDING_COLS], rotation=20)
        ax.set_ylim(0, 1)
        ax.axhline(0.9, color='red', linestyle='--', linewidth=1)
        ax.set_title(f'{title} (pooled, n={len(df)} clips)')
        ax.legend()
    plt.tight_layout()
    plt.show()


: 

## 11. Bonus: autoencoder reconstruction-error as a pneumothorax anomaly score

An empirical test of the alternative considered (and set aside) in `Models-Registry.md`: train an
autoencoder on **normal**-lung frames only, then score any frame by how badly it reconstructs
(**high reconstruction error = "doesn't look like normal lung"**). The documented reasoning
against using this for pneumothorax was that its only sign (absent lung sliding) is a *motion*
pattern — invisible in any single static frame, so a per-frame anomaly score has nothing to find.
This section runs the actual experiment instead of just asserting that, using the manifest's
`finding_pneumothorax` / `finding_normal` labels (unused elsewhere in this notebook — pneumothorax
is excluded from the classifier per the notebook intro).

Independent of the classifier pipeline above (one split, not CV — this is a diagnostic side
experiment, not a model being shipped):
- **Train** the autoencoder only on clips flagged `finding_normal` (held out 80/20 within that group).
- **Evaluate** reconstruction error on the held-out normal clips vs. the `finding_pneumothorax`
  clips: if the idea worked, pneumothorax frames should reconstruct worse (higher error) than
  normal frames. AUROC treats error as the score, pneumothorax as the positive class.


In [ ]:
AE_IMG_SIZE = 64          # small on purpose -- from-scratch autoencoder on ~40 normal training clips
AE_FRAMES_PER_CLIP = 16   # more frames/clip than the classifier uses (8) since this dataset is much smaller
AE_EPOCHS = 40

normal_df = df[df['finding_normal']].reset_index(drop=True)
pneumo_df = df[df['finding_pneumothorax']].reset_index(drop=True)
print(f'Normal clips: {len(normal_df)}, Pneumothorax clips: {len(pneumo_df)}')

ae_train_df, ae_val_df = train_test_split(normal_df, test_size=0.2, random_state=42)
print(f'AE train (normal): {len(ae_train_df)} clips, AE val (normal, held out): {len(ae_val_df)} clips')


def load_ae_frames(manifest_df, n_frames):
    """Grayscale, AE_IMG_SIZE x AE_IMG_SIZE, [0, 1]-normalized frames, tagged with clip_pos for
    per-clip error aggregation later. Reuses extract_frames (section 4) -- same decoding, just a
    lighter/smaller representation since the autoencoder doesn't need ImageNet-style RGB input."""
    frames, clip_ids = [], []
    for clip_pos, (_, row) in enumerate(manifest_df.iterrows()):
        filepath = DATA_ROOT / row['filepath']
        if not filepath.exists():
            print('Missing file, skipping:', filepath)
            continue
        for frame in extract_frames(filepath, row['media_type'], n_frames):
            gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
            gray = cv2.resize(gray, (AE_IMG_SIZE, AE_IMG_SIZE))
            frames.append(gray.astype(np.float32) / 255.0)
            clip_ids.append(clip_pos)
    x = torch.from_numpy(np.stack(frames)).unsqueeze(1)  # (N, 1, H, W)
    clip_ids = np.array(clip_ids)
    return x, clip_ids


ae_train_x, _ = load_ae_frames(ae_train_df, AE_FRAMES_PER_CLIP)
ae_val_x, ae_val_clip_ids = load_ae_frames(ae_val_df, AE_FRAMES_PER_CLIP)
pneumo_x, pneumo_clip_ids = load_ae_frames(pneumo_df, AE_FRAMES_PER_CLIP)
print(f'AE train frames: {len(ae_train_x)}, AE val (normal) frames: {len(ae_val_x)}, pneumothorax frames: {len(pneumo_x)}')


: 

In [ ]:
from torch.utils.data import TensorDataset


class ConvAutoencoder(nn.Module):
    """Small from-scratch conv autoencoder -- deliberately lightweight (only ~35 training clips
    here), not a pretrained backbone. Bottleneck at 8x8xlatent_channels for a 64x64 input."""
    def __init__(self, latent_channels=16):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=2, padding=1), nn.ReLU(inplace=True),   # 64 -> 32
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(inplace=True),  # 32 -> 16
            nn.Conv2d(64, latent_channels, 3, stride=2, padding=1), nn.ReLU(inplace=True),  # 16 -> 8
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(latent_channels, 64, 3, stride=2, padding=1, output_padding=1), nn.ReLU(inplace=True),  # 8 -> 16
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1), nn.ReLU(inplace=True),  # 16 -> 32
            nn.ConvTranspose2d(32, 1, 3, stride=2, padding=1, output_padding=1), nn.Sigmoid(),  # 32 -> 64
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))


ae_model = ConvAutoencoder().to(device)
ae_optimizer = torch.optim.Adam(ae_model.parameters(), lr=1e-3)
ae_criterion = nn.MSELoss()
ae_train_loader = DataLoader(TensorDataset(ae_train_x), batch_size=32, shuffle=True)

ae_loss_history = []
for epoch in range(AE_EPOCHS):
    ae_model.train()
    total_loss = 0.0
    for (batch,) in ae_train_loader:
        batch = batch.to(device)
        recon = ae_model(batch)
        loss = ae_criterion(recon, batch)
        ae_optimizer.zero_grad()
        loss.backward()
        ae_optimizer.step()
        total_loss += loss.item() * batch.size(0)
    epoch_loss = total_loss / len(ae_train_x)
    ae_loss_history.append(epoch_loss)
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f'AE epoch {epoch+1:02d}/{AE_EPOCHS} | recon_loss={epoch_loss:.5f}')

plt.figure(figsize=(6, 4))
plt.plot(ae_loss_history)
plt.xlabel('Epoch'); plt.ylabel('Train reconstruction MSE')
plt.title('Autoencoder training loss (normal-lung frames only)')
plt.grid(alpha=0.3)
plt.show()


: 

In [ ]:
from sklearn.metrics import roc_curve


def per_frame_recon_error(model, x, batch_size=64):
    model.eval()
    errors = []
    with torch.no_grad():
        for i in range(0, len(x), batch_size):
            batch = x[i:i + batch_size].to(device)
            recon = model(batch)
            err = ((recon - batch) ** 2).mean(dim=[1, 2, 3])  # per-frame MSE
            errors.append(err.cpu().numpy())
    return np.concatenate(errors)


def per_clip_recon_error(model, x, clip_ids):
    frame_errors = per_frame_recon_error(model, x)
    n_clips = clip_ids.max() + 1
    return np.array([frame_errors[clip_ids == c].mean() for c in range(n_clips)])


normal_clip_errors = per_clip_recon_error(ae_model, ae_val_x, ae_val_clip_ids)
pneumo_clip_errors = per_clip_recon_error(ae_model, pneumo_x, pneumo_clip_ids)

scores = np.concatenate([normal_clip_errors, pneumo_clip_errors])
labels = np.concatenate([np.zeros(len(normal_clip_errors)), np.ones(len(pneumo_clip_errors))])
auroc = roc_auc_score(labels, scores)
print(f'Held-out normal clips: {len(normal_clip_errors)}, pneumothorax clips: {len(pneumo_clip_errors)}')
print(f'Reconstruction-error AUROC for pneumothorax vs normal: {auroc:.3f} (0.5 = no better than chance)')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(normal_clip_errors, bins=12, alpha=0.6, label='normal (held out)', color='steelblue')
axes[0].hist(pneumo_clip_errors, bins=12, alpha=0.6, label='pneumothorax', color='indianred')
axes[0].set_xlabel('Mean per-clip reconstruction error')
axes[0].set_ylabel('Clips')
axes[0].set_title('Reconstruction error by group')
axes[0].legend()

fpr, tpr, _ = roc_curve(labels, scores)
axes[1].plot(fpr, tpr, label=f'AUROC={auroc:.3f}')
axes[1].plot([0, 1], [0, 1], linestyle='--', color='gray')
axes[1].set_xlabel('False positive rate')
axes[1].set_ylabel('True positive rate')
axes[1].set_title('ROC — reconstruction error as pneumothorax score')
axes[1].legend()
plt.tight_layout()
plt.show()

print()
if auroc < 0.65:
    print("Result matches the documented prediction: reconstruction error barely separates pneumothorax")
    print("from normal lung, because the actual sign (absent lung sliding) is a motion pattern that a")
    print("single static frame can't carry -- a frame from a pneumothorax clip can look identical to a")
    print("normal one at the instant it was sampled. This empirically supports using the classical")
    print("motion-variance/optical-flow feature instead (Models-Registry.md), not this autoencoder branch.")
else:
    print("Higher separation than the motion-signal argument predicted -- worth a second look before")
    print("trusting it: check whether this is a real texture cue (different sonographic features happen")
    print("to correlate with pneumothorax in this dataset) or a confound (different source/probe/")
    print("preprocessing between the pneumothorax and normal subsets, rather than pneumothorax itself).")


: 